# A minimal IMD on the sphere

**Point cloud → proximity graph → drift and covariance → continuous trajectories.**

Everything numerical lives in `imd_tutorialsrc/` and uses NumPy/SciPy, with no training and no imports from the main repository. All figures use the sphere-only copy of `plot_diffusion_3D` from the requested `implicit-manifolds/src/plotting.py`.

We target Brownian motion with generator $\frac12\Delta_{S^2}$. The simulator sees only the point cloud; sphere geometry is used to generate data and inspect radial error, never to project simulated paths.

If needed, install dependencies in this notebook's environment using `%pip install -r requirements.txt` (from this directory).

In [1]:
from pathlib import Path
import sys
import numpy as np
from scipy import sparse
from scipy.sparse.csgraph import connected_components

# Works when Jupyter starts in this directory or at the repository root.
HERE = next(p for root in (Path.cwd(), *Path.cwd().parents)
            for p in (root, root / "src")
            if (p / "src").is_dir())
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from src import (
    sample_uniform_sphere, sample_vmf_sphere,
    proximity_graph, graph_generator, drift_and_cdc,
    make_coefficient_lookup, euler_maruyama,
)
from src.plotting import plot_diffusion_3D

## 1. Load or sample observations

Choose `"uniform"` or `"vmf"`. The von Mises–Fisher distribution has density proportional to $\exp(\kappa\mu^\top x)$ on the unit sphere; larger $\kappa$ puts more points near $\mu$. Start with uniform data for the clearest example. Each choice is cached locally.

In [2]:
DISTRIBUTION = "uniform"  # change to "vmf" to try nonuniform observations
N, DATA_SEED = 8000, 0
KAPPA = 1.0  # vmf mean direction is fixed to the north pole here

if DISTRIBUTION not in {"uniform", "vmf"}:
    raise ValueError("Choose 'uniform' or 'vmf'.")
cache_dir = HERE / "data"
cache_dir.mkdir(exist_ok=True)
tag = DISTRIBUTION if DISTRIBUTION == "uniform" else f"vmf_kappa{KAPPA:g}_north"
cache = cache_dir / f"sphere_{tag}_n{N}_seed{DATA_SEED}.npy"
if cache.exists():
    X = np.load(cache, allow_pickle=False)
    print(f"Loaded {cache.name}")
else:
    X = (sample_uniform_sphere(N, seed=DATA_SEED) if DISTRIBUTION == "uniform"
         else sample_vmf_sphere(N, kappa=KAPPA, seed=DATA_SEED))
    np.save(cache, X)
    print(f"Saved {cache.name}")
assert X.shape == (N, 3) and np.allclose(np.linalg.norm(X, axis=1), 1)
fig = plot_diffusion_3D(X, mode="sphere", show_scene=False)
fig.update_layout(title=f"{N:,} {DISTRIBUTION} observations", margin=dict(t=45))
fig.show()

Loaded sphere_uniform_n8000_seed0.npy


## 2. Build and visualize a proximity graph

Connect points whose **ambient Euclidean distance** is at most $r$. We set $r$ to the median distance to the 40th other point. This is a radius graph, not a graph where every vertex has exactly 40 neighbors. Self-loops are included in the normalization but not drawn.

The plot colors vertices by degree and draws a fixed random subset of edges for readability. The computation uses the entire graph. If the graph is disconnected, increase `GRAPH_K`; sparse regions in a concentrated vMF cloud may need more observations.

In [3]:
GRAPH_K = 40
W, radius = proximity_graph(X, k=GRAPH_K)
degree = np.asarray(W.sum(axis=1)).ravel() - 1
n_components = connected_components(W, directed=False, return_labels=False)
print(f"Radius = {radius:.4f}; components = {n_components}; mean degree = {degree.mean():.1f}")

# NaN-separated segments let the same plotting helper display graph edges.
row, col = sparse.triu(W, k=1).nonzero()
chosen = np.random.default_rng(1).choice(len(row), min(2000, len(row)), replace=False)
segments = np.full((3 * len(chosen), 3), np.nan)
segments[0::3], segments[1::3] = X[row[chosen]], X[col[chosen]]
fig = plot_diffusion_3D(
    X, point_values=degree, trajectories=segments,
    trajectory_style=dict(color="rgba(60,90,140,0.20)", width=1),
    marker_style=dict(size=2, colorscale="Viridis", colorbar=dict(title="Degree")),
    show_scene=False, show_legend=False,
)
fig.update_layout(title=f"Radius graph ({len(chosen):,} of {len(row):,} edges shown)",
                  margin=dict(t=45))
fig.show()

Radius = 0.1410; components = 1; mean degree = 39.8


## 3. Turn the graph into drift and covariance

Apply density correction $\widetilde W_{ij}=W_{ij}/(q_iq_j)^\alpha$, with $q_i=\sum_jW_{ij}$, and row-normalize to get $P$. With intrinsic dimension $m=2$, the binary-ball kernel gives
$$A=\frac{m+2}{r^2}(P-I)=\frac4{r^2}(P-I).$$
The negative-semidefinite generator $A$ targets $\frac12\Delta$ when $\alpha=1$ removes observation-density drift (asymptotically). Uniform data also permit $\alpha=0$.

We compute directly from coordinate products:
$$b^a=A x^a,\qquad \Gamma^{ab}=A(x^a x^b)-x^aA x^b-x^bA x^a.$$
Here $\Gamma$ is the **noise covariance**, i.e. the full product-rule defect without the optional factor of $1/2$ in some CDC conventions. On the exact unit sphere, $b(x)=-x$ and $\Gamma(x)=I-xx^\top$; these identities are diagnostics only.

In [4]:
ALPHA = 1.0
A = graph_generator(W, radius, alpha=ALPHA, intrinsic_dim=2)
b, Gamma = drift_and_cdc(X, A)
print("drift:", b.shape, "CDC:", Gamma.shape)
print(f"Generator row-sum error: {np.abs(np.asarray(A.sum(axis=1))).max():.2e}")
print(f"Mean covariance trace: {np.trace(Gamma, axis1=1, axis2=2).mean():.3f} (sphere target: 2)")

drift: (8000, 3) CDC: (8000, 3, 3)
Generator row-sum error: 1.59e-13
Mean covariance trace: 1.942 (sphere target: 2)


## 4. Run and visualize an IMD

Interpolate $b$ and $\Gamma$ at the current ambient state using 20 nearby observations, then take Euler–Maruyama steps:
$$Z_{n+1}=Z_n+b(Z_n)\Delta t+\sqrt{\Delta t}\,\Sigma(Z_n)\xi_n,
\quad\xi_n\sim\mathcal N(0,I),\quad\Sigma\Sigma^\top=\Gamma.$$

We discard estimated covariance eigenvalues below 1% of the largest to reduce normal noise. This regularization, the finite graph, and the time step all introduce approximation error. Paths are **not projected back to the sphere** and can drift off it.

Six trajectories start at the observation nearest the north pole. They share their start but have independently generated Gaussian increments.

In [5]:
N_PATHS, N_STEPS, DT = 6, 10000, 1e-4
start = X[np.argmax(X[:, 2])]
initial = np.repeat(start[None, :], N_PATHS, axis=0)
lookup = make_coefficient_lookup(X, b, Gamma, k=20)
paths = euler_maruyama(initial, lookup, dt=DT, n_steps=N_STEPS, seed=7)
radial_error = np.abs(np.linalg.norm(paths, axis=-1) - 1)
print(f"Paths: {paths.shape}; horizon = {N_STEPS * DT:g}")
print(f"Mean / max radial deviation: {radial_error.mean():.4f} / {radial_error.max():.4f}")

fig = plot_diffusion_3D(
    X, mode="sphere", trajectories=list(paths),
    marker_style=dict(size=2, color="lightgray", opacity=0.25),
    show_scene=False,
)
fig.update_layout(title="Continuous IMD trajectories (no surface projection)",
                  margin=dict(t=45))
fig.show()

Paths: (6, 10001, 3); horizon = 1
Mean / max radial deviation: 0.0308 / 0.1131


### Try changing one thing

- Switch the observations to `"vmf"` and compare `ALPHA = 0.0` with `1.0`.
- Increase `N` to refine the point cloud.
- Halve `DT` and double `N_STEPS` to keep the same horizon.
- Change `eigenvalue_cutoff=0.0` in the simulator call to retain all covariance directions.

The comparison is qualitative: this small example is an implementation tutorial, not a convergence benchmark.